# Stage 2 — BindCraft cloud smoke test (EGFR challenge)

Purpose: distinguish **ENVIRONMENT FAILURE** from **EGFR TARGET FAILURE**.

1. Smallest official example: PDL1, hotspot 56, 65 aa, cap 1 relaxed trajectory.
2. Micro EGFR run: Domain III crop 310-481, approved envelope B, conservative hotspot `390,393,399,421,424,431`, 80 aa, cap 3 relaxed trajectories.

Rules baked into this notebook: no target mutation, no hard-target settings, official default 4stage-multimer advanced/filter files unchanged (only `max_trajectories`), no paid compute.

**Environment note (attempt 001 failed on Colab's rolling image):** Colab currently ships Python 3.13.15 + JAX 0.11.1, where `jax.lib.xla_bridge` no longer exists (removed in JAX 0.8.0); ColabDesign's `clear_mem()` then crashes inside the official PDL1 example before any trajectory. This notebook therefore builds an **isolated Miniforge environment that reproduces the current official `install_bindcraft.sh` exactly**: Python 3.10, conda-forge/nvidia `jax=0.6.0 jaxlib=0.6.0=*cuda*` (CUDA 12.6), `numpy<2`, `flax<0.10`, ColabDesign installed `--no-deps` at a pinned commit. The Colab system Python is never used for compute. Evidence and versions: `reports/stage2_environment_failure_001.md`.

**Before running:** Runtime -> Change runtime type -> T4 GPU. If a previous attempt is open, do **Runtime -> Disconnect and delete runtime** (factory reset) first so the old broken JAX cannot leak in. Then run cells top to bottom; do not change any numbers. If a cell OOMs twice, stop and report.

Pinned: BindCraft `7713aa0d0d351e4117a8befeb8541f3a8ebd3368`; ColabDesign `e31a56fe1d9b4de25c8697f3a28b75892941cc72`; weights `alphafold_params_2022-12-06.tar`.

## Cell 1 — baseline hardware metadata (Colab host kernel)

In [ ]:
import json, os, platform, shutil, subprocess, sys, time

def sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()

def gpu_query(field):
    return sh(f"nvidia-smi --query-gpu={field} --format=csv,noheader,nounits 2>/dev/null")

env_pre = {
    'colab_system_python': sys.version.split()[0],
    'platform': platform.platform(),
    'gpu_name': gpu_query('name') or 'NO NVIDIA GPU VISIBLE',
    'gpu_vram_total_mib': int(x) if (x := gpu_query('memory.total')) else None,
    'gpu_vram_free_mib_at_start': int(x) if (x := gpu_query('memory.free')) else None,
    'host_ram_total_gb': round(os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 1e9, 1),
    'disk_free_gb_content': round(shutil.disk_usage('/content').free / 1e9, 1),
    'colab_nvcc': sh('nvcc --version | tail -1') or None,
    'nvidia_driver': sh('nvidia-smi --query-gpu=driver_version --format=csv,noheader 2>/dev/null') or None,
}
print(json.dumps(env_pre, indent=2))
assert env_pre['gpu_name'] != 'NO NVIDIA GPU VISIBLE', 'GPU runtime not enabled! Runtime -> Change runtime type -> GPU, then reconnect.'

## Cell 2 — build isolated BindCraft environment (Python 3.10 + JAX 0.6.0 CUDA)

Reproduces upstream `install_bindcraft.sh` (BindCraft 7713aa0) on Colab without touching the preinstalled Python 3.13/JAX 0.11.1. First run downloads several GB (10-25 min); re-runs reuse `/content/bindcraft_env` (lost on factory reset, which is expected).

In [ ]:
import os, subprocess

ENV_PREFIX = '/content/bindcraft_env'
MF_PREFIX = '/content/miniforge3'
MAMBA = f'{MF_PREFIX}/bin/mamba'
BINDPY = f'{ENV_PREFIX}/bin/python'
COLABDESIGN_PIN = 'e31a56fe1d9b4de25c8697f3a28b75892941cc72'
CUDA_OVERRIDE = '12.6'  # conda-forge jaxlib 0.6.0 cuda build requires cuda-version >=12.6,<13

def runv(argv, env=None, quiet=False):
    p = subprocess.run(argv, capture_output=True, text=True, env=env)
    if p.returncode != 0:
        print('FAILED:', ' '.join(argv))
        print(p.stdout[-4000:]); print(p.stderr[-4000:])
        raise SystemExit(p.returncode)
    if not quiet:
        print('ok:', ' '.join(str(a) for a in argv[:4]))
    return p

if not os.path.exists(f'{ENV_PREFIX}/.env_ready'):
    if not os.path.exists(MAMBA):
        print('installing Miniforge (isolated; Colab system Python stays untouched)')
        runv(['wget', '-q',
              'https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh',
              '-O', '/tmp/miniforge.sh'])
        runv(['bash', '/tmp/miniforge.sh', '-b', '-p', MF_PREFIX])
    # exact package set of upstream install_bindcraft.sh; jax/jaxlib pinned to
    # the upper edge of the official <=0.6.0 range, CUDA 12.6 build selected
    spec = ['python=3.10', 'pip', 'pandas', 'matplotlib', 'numpy<2.0.0',
            'biopython', 'scipy', 'pdbfixer', 'seaborn', 'libgfortran5',
            'tqdm', 'jupyter', 'ffmpeg', 'fsspec', 'py3dmol', 'chex',
            'dm-haiku', 'flax<0.10.0', 'dm-tree', 'joblib', 'ml-collections',
            'immutabledict', 'optax', 'jax=0.6.0', 'jaxlib=0.6.0=*cuda*',
            'cuda-nvcc', 'cudnn']
    env = dict(os.environ, CONDA_OVERRIDE_CUDA=CUDA_OVERRIDE)
    print('creating conda env from upstream spec (this is the long step)...')
    runv([MAMBA, 'create', '-y', '-p', ENV_PREFIX,
          '-c', 'conda-forge', '-c', 'nvidia', *spec], env=env)
    print('installing ColabDesign pinned commit with --no-deps (upstream pattern)')
    runv([f'{ENV_PREFIX}/bin/pip', 'install', '--no-deps',
          f'git+https://github.com/sokrypton/ColabDesign.git@{COLABDESIGN_PIN}'])
    print('installing PyRosetta wheel (~1.6 GB; academic/non-commercial license)')
    runv([f'{ENV_PREFIX}/bin/pip', 'install', 'pyrosetta', '--find-links',
          'https://west.rosettacommons.org/pyrosetta/quarterly/release.cxx11thread.serialization'])
    runv([BINDPY, '-c', 'import sys; assert sys.version_info[:2] == (3, 10)'], quiet=True)
    open(f'{ENV_PREFIX}/.env_ready', 'w').write('ok\n')
    print('environment ready')
else:
    print('reusing existing', ENV_PREFIX)
print(runv([BINDPY, '-V'], quiet=True).stdout.strip())

## Cell 3 — BindCraft source at the pinned commit

In [ ]:
PINNED = '7713aa0d0d351e4117a8befeb8541f3a8ebd3368'
if not os.path.isfile('/content/bindcraft/.pinned_sha'):
    print('cloning BindCraft pinned commit')
    runv(['git', 'clone', 'https://github.com/martinpacesa/BindCraft', '/content/bindcraft'])
    runv(['git', '-C', '/content/bindcraft', 'checkout', PINNED])
    os.chmod('/content/bindcraft/functions/dssp', 0o755)
    os.chmod('/content/bindcraft/functions/DAlphaBall.gcc', 0o755)
    open('/content/bindcraft/.pinned_sha', 'w').write(PINNED)
sha = runv(['git', '-C', '/content/bindcraft', 'rev-parse', 'HEAD'], quiet=True).stdout.strip()
assert sha == PINNED, f'unexpected checkout {sha}'
print('BindCraft at', sha)

## Cell 4 — mount Google Drive (results persist beyond the Colab session)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
RUNROOT = '/content/drive/MyDrive/BindCraft/stage2_smoke'
os.makedirs(RUNROOT, exist_ok=True)
print('results ->', RUNROOT)

## Cell 5 — upload target crop PDB, analyzer, and pre-flight gate

Upload three files from this repo: `data/processed/6ARU_chainA_domain3_310-481.pdb`, `scripts/analyze_bindcraft_run.py`, `scripts/bindcraft_preflight.py`.


In [ ]:
from google.colab import files
os.makedirs('/content/egfr', exist_ok=True)
up = files.upload()
for name, data in up.items():
    dest = f'/content/egfr/{name}' if name.endswith('.pdb') else f'/content/{name}'
    open(dest, 'wb').write(data)
    print('saved', dest)

for required in ('/content/egfr/6ARU_chainA_domain3_310-481.pdb',
                 '/content/analyze_bindcraft_run.py',
                 '/content/bindcraft_preflight.py'):
    assert os.path.isfile(required), f'missing upload: {required}'

# verify crop: 172 residues, UniProt numbering 310-481, hotspot identities
res = {}
for line in open('/content/egfr/6ARU_chainA_domain3_310-481.pdb'):
    if line.startswith('ATOM') and line[12:16].strip() == 'CA':
        res[int(line[22:26])] = line[17:20].strip()
aa3 = {'Q': 'GLN', 'E': 'GLU', 'D': 'ASP', 'K': 'LYS', 'R': 'ARG', 'N': 'ASN', 'T': 'THR'}
expect = {390: 'Q', 391: 'E', 393: 'D', 396: 'K', 399: 'K', 421: 'E', 422: 'N',
          424: 'E', 427: 'R', 429: 'R', 430: 'T', 431: 'K'}
assert sorted(res) == list(range(310, 482)), f'crop numbering wrong: {len(res)} residues'
assert all(res[p] == aa3[a] for p, a in expect.items()), 'hotspot identity mismatch'
print('crop verified:', len(res), 'residues; all 12 exposed-residue identities match')

## Cell 6 — PRE-FLIGHT compatibility gate (hard stop before any weight download)

Runs inside the isolated Python 3.10 env: version policy (Python 3.10, jax/jaxlib within 0.4-0.6.0, numpy<2, flax<0.10), `colabdesign.clear_mem()`, direct `jax.lib.xla_bridge.get_backend()`, GPU detection, a 2048x2048 GPU matmul, and PyRosetta. If anything fails the cell raises — do not continue; send back `preflight.json`.

In [ ]:
job_env = dict(os.environ)
job_env['LD_LIBRARY_PATH'] = '/content/bindcraft_env/lib:' + job_env.get('LD_LIBRARY_PATH', '')
job_env['PATH'] = '/content/bindcraft_env/bin:' + job_env.get('PATH', '')

pf_path = f'{RUNROOT}/preflight.json'
pfp = subprocess.run([BINDPY, '/content/bindcraft_preflight.py', '--out', pf_path],
                     capture_output=True, text=True, env=job_env)
print(pfp.stdout)
if pfp.stderr.strip():
    print('--- stderr ---'); print(pfp.stderr[-3000:])
pf = json.load(open(pf_path))
assert pfp.returncode == 0 and pf['passed'], \
    'PRE-FLIGHT FAILED -> environment failure; stop here, do not download weights or run PDL1'
assert pf['gpu']['platform'] == 'gpu', 'JAX does not see a GPU'
assert pf['checks'] and all(c['ok'] for c in pf['checks'])
print('PRE-FLIGHT PASSED on', pf['gpu']['device_kind'],
      '| jax', pf['versions']['jax'], '| cuda backend',
      pf['xla_bridge']['platform_version'])

## Cell 7 — Validate / fetch official AlphaFold2 model weights + final env metadata

Self-contained inline harness (attempt-004): skips if the exact official **15**-.npz set (5 base + 5 pTM + 5 Multimer-v3) is already present; otherwise ensures `wget`, downloads/resumes `alphafold_params_2022-12-06.tar` with an observed child process (PID, size-based progress, return code, `download.log`), fails fast on non-zero exit with the real log tail, checks tar integrity, extracts, validates the exact 15-file set, and only then writes `done.txt`. Any failure stops here — report the log tail.

In [ ]:
# ============================================================
# Cell 7 — Validate / fetch official AlphaFold2 model weights
#          + record final environment metadata
#
# Official alphafold_params_2022-12-06.tar contains:
#   5 base models
#   5 pTM models
#   5 Multimer-v3 models
# = 15 .npz parameter files
# ============================================================

import os
import glob
import json
import shutil
import subprocess
import time
from pathlib import Path


# ------------------------------------------------------------
# Paths / constants
# ------------------------------------------------------------

PARAMS_DIR = Path("/content/bindcraft/params")
PARAMS_DIR.mkdir(parents=True, exist_ok=True)

AF2_URL = (
    "https://storage.googleapis.com/alphafold/"
    "alphafold_params_2022-12-06.tar"
)

ARCHIVE = PARAMS_DIR / "alphafold_params_2022-12-06.tar"
DONE_FILE = PARAMS_DIR / "done.txt"
DOWNLOAD_LOG = PARAMS_DIR / "download.log"

RUNROOT = globals().get(
    "RUNROOT",
    "/content/drive/MyDrive/BindCraft/stage2_smoke"
)
os.makedirs(RUNROOT, exist_ok=True)

BINDPY = globals().get(
    "BINDPY",
    "/content/bindcraft_env/bin/python"
)

ENV_PREFIX = globals().get(
    "ENV_PREFIX",
    "/content/bindcraft_env"
)

assert os.path.isfile(BINDPY), (
    f"isolated BindCraft Python missing: {BINDPY}"
)


# ------------------------------------------------------------
# Exact expected files from the official 2022-12-06 package
# ------------------------------------------------------------

REQUIRED_WEIGHTS = []

for i in range(1, 6):
    REQUIRED_WEIGHTS.extend(
        [
            f"params_model_{i}.npz",
            f"params_model_{i}_ptm.npz",
            f"params_model_{i}_multimer_v3.npz",
        ]
    )

REQUIRED_SET = set(REQUIRED_WEIGHTS)

assert len(REQUIRED_SET) == 15


def inspect_weights(verbose=True):
    """
    Return:
        ok, present, missing, unexpected
    """
    present = {
        p.name
        for p in PARAMS_DIR.glob("*.npz")
        if p.is_file()
    }

    missing = sorted(REQUIRED_SET - present)
    unexpected = sorted(present - REQUIRED_SET)

    ok = (
        len(present) == 15
        and not missing
        and not unexpected
    )

    if verbose:
        print(f"AF2 .npz files present: {len(present)}")

        if missing:
            print("Missing required files:")
            for x in missing:
                print("  MISSING:", x)

        if unexpected:
            print("Unexpected .npz files:")
            for x in unexpected:
                print("  EXTRA:", x)

        if ok:
            print(
                "OK: exact official 15-file "
                "AlphaFold parameter set is present"
            )

    return ok, present, missing, unexpected


# ------------------------------------------------------------
# Step 1 — reuse existing extracted weights if valid
# ------------------------------------------------------------

weights_ok, present, missing, unexpected = inspect_weights()

if weights_ok:

    print("\nExisting AlphaFold weights are already valid.")
    print("Skipping the 5.3 GB download.")

    DONE_FILE.write_text(
        "validated: official 15-file "
        "alphafold_params_2022-12-06 set\n"
    )

else:

    print("\nExisting weights are incomplete or invalid.")

    # Never trust stale done.txt.
    if DONE_FILE.exists():
        print("Removing stale done.txt")
        DONE_FILE.unlink()

    # --------------------------------------------------------
    # Step 2 — ensure wget exists
    #
    # BindCraft upstream uses wget for this exact archive.
    # --------------------------------------------------------

    wget_path = shutil.which("wget")

    if wget_path is None:
        print("wget absent; installing via apt-get ...")

        update = subprocess.run(
            ["apt-get", "update", "-qq"],
            text=True,
        )

        assert update.returncode == 0, (
            "apt-get update failed"
        )

        install = subprocess.run(
            ["apt-get", "install", "-y", "wget"],
            text=True,
        )

        assert install.returncode == 0, (
            "wget installation failed"
        )

        wget_path = shutil.which("wget")

    assert wget_path is not None, (
        "wget unavailable after installation attempt"
    )

    print("wget:", wget_path)

    # --------------------------------------------------------
    # Step 3 — download / resume archive
    # --------------------------------------------------------

    if ARCHIVE.exists():
        size_gib = ARCHIVE.stat().st_size / (1024 ** 3)

        print(
            f"Existing partial/full archive found: "
            f"{size_gib:.2f} GiB"
        )
        print("wget -c will attempt resume/reuse.")

    print("\nDownloading/resuming official AF2 weights:")
    print(AF2_URL)
    print("Destination:", ARCHIVE)

    cmd = [
        wget_path,
        "-c",
        "--progress=dot:giga",
        "-O",
        str(ARCHIVE),
        AF2_URL,
    ]

    start_time = time.time()

    # Log the actual downloader output.
    with open(DOWNLOAD_LOG, "a", buffering=1) as log:

        log.write(
            "\n\n===== AF2 DOWNLOAD START =====\n"
        )
        log.write("COMMAND: " + " ".join(cmd) + "\n")
        log.flush()

        proc = subprocess.Popen(
            cmd,
            stdout=log,
            stderr=subprocess.STDOUT,
            text=True,
        )

        print("download PID:", proc.pid)
        print(
            "download log:",
            str(DOWNLOAD_LOG),
        )

        previous_size = -1

        # No arbitrary 30-minute timeout.
        # Observe the actual child process.
        while proc.poll() is None:

            if ARCHIVE.exists():
                current_size = ARCHIVE.stat().st_size

                if current_size != previous_size:
                    gib = current_size / (1024 ** 3)

                    print(
                        f"downloaded archive size: "
                        f"{gib:.2f} GiB"
                    )

                    previous_size = current_size

            time.sleep(15)

        returncode = proc.returncode

    elapsed = time.time() - start_time

    print(
        f"download process finished in "
        f"{elapsed / 60:.1f} min"
    )
    print("wget return code:", returncode)

    if returncode != 0:

        print("\n--- download.log tail ---")

        try:
            with open(
                DOWNLOAD_LOG,
                "r",
                errors="replace"
            ) as f:
                tail = f.readlines()[-80:]

            print("".join(tail))

        except Exception as e:
            print(
                "Could not read download log:",
                repr(e)
            )

        raise RuntimeError(
            "AlphaFold weight download failed; "
            "see download.log above"
        )

    assert ARCHIVE.is_file(), (
        "wget exited 0 but archive is missing"
    )

    archive_size = ARCHIVE.stat().st_size

    print(
        "archive size:",
        f"{archive_size / (1024 ** 3):.2f} GiB"
    )

    # --------------------------------------------------------
    # Step 4 — verify tar integrity BEFORE extraction
    # --------------------------------------------------------

    print("\nChecking tar archive integrity ...")

    tar_check = subprocess.run(
        [
            "tar",
            "-tf",
            str(ARCHIVE),
        ],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.PIPE,
        text=True,
    )

    if tar_check.returncode != 0:

        print(
            tar_check.stderr[-4000:]
        )

        raise RuntimeError(
            "Downloaded AlphaFold archive is "
            "corrupt or incomplete"
        )

    print("tar integrity: PASS")

    # --------------------------------------------------------
    # Step 5 — extract
    # --------------------------------------------------------

    print("\nExtracting AlphaFold parameters ...")

    extract = subprocess.run(
        [
            "tar",
            "-xf",
            str(ARCHIVE),
            "-C",
            str(PARAMS_DIR),
        ],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
    )

    if extract.returncode != 0:

        print(extract.stderr[-4000:])

        raise RuntimeError(
            "AlphaFold parameter extraction failed"
        )

    print("extraction: PASS")

    # --------------------------------------------------------
    # Step 6 — validate exact official 15-file set
    # --------------------------------------------------------

    weights_ok, present, missing, unexpected = (
        inspect_weights()
    )

    if not weights_ok:
        raise RuntimeError(
            "Archive extracted, but the exact official "
            "15-file AlphaFold parameter set was not found"
        )

    # Only create done.txt AFTER complete validation.
    DONE_FILE.write_text(
        "validated: official 15-file "
        "alphafold_params_2022-12-06 set\n"
    )

    print("done.txt created")

    # Safe to delete archive only after successful validation.
    try:
        ARCHIVE.unlink()
        print(
            "validated archive removed to save disk space"
        )
    except Exception as e:
        print(
            "WARNING: could not remove archive:",
            repr(e)
        )


# ------------------------------------------------------------
# FINAL HARD WEIGHT GATE
# ------------------------------------------------------------

weights_ok, present, missing, unexpected = inspect_weights()

assert weights_ok, (
    "FINAL AF2 WEIGHT VALIDATION FAILED"
)

assert DONE_FILE.is_file(), (
    "done.txt missing after successful validation"
)

weight_files = sorted(present)

print("\nFINAL AF2 WEIGHT FILES:")

for name in weight_files:
    size_mb = (
        (PARAMS_DIR / name).stat().st_size
        / (1024 ** 2)
    )

    print(
        f"  {name:40s} "
        f"{size_mb:8.1f} MiB"
    )

print(
    "\nOK: 15 required AlphaFold .npz files validated"
)


# ============================================================
# Record final environment metadata
# This replaces the old Cell 7 metadata tail as well.
# ============================================================

print("\nRecording final environment metadata ...")


def capture(cmd):
    p = subprocess.run(
        cmd,
        capture_output=True,
        text=True,
    )

    return p.stdout.strip()


# ------------------------------------------------------------
# Reload preflight result from the persistent Stage-2 directory
# ------------------------------------------------------------

PREFLIGHT_PATH = os.path.join(
    RUNROOT,
    "preflight.json"
)

assert os.path.isfile(PREFLIGHT_PATH), (
    f"preflight.json missing: {PREFLIGHT_PATH}"
)

with open(PREFLIGHT_PATH) as f:
    pf = json.load(f)

assert pf.get("passed") is True, (
    "preflight.json does not report passed=true"
)


# ------------------------------------------------------------
# BindCraft commit
# ------------------------------------------------------------

bindcraft_commit = capture(
    [
        "git",
        "-C",
        "/content/bindcraft",
        "rev-parse",
        "HEAD",
    ]
)


# ------------------------------------------------------------
# pip freeze inside isolated Python environment
# ------------------------------------------------------------

pip_freeze = capture(
    [
        BINDPY,
        "-m",
        "pip",
        "freeze",
    ]
)


# ------------------------------------------------------------
# conda/mamba package list if available
# ------------------------------------------------------------

conda_candidates = [
    "/content/miniforge/bin/mamba",
    "/content/miniforge/bin/conda",
]

conda_tool = next(
    (
        x
        for x in conda_candidates
        if os.path.isfile(x)
    ),
    None,
)

if conda_tool:
    conda_list = capture(
        [
            conda_tool,
            "list",
            "-p",
            ENV_PREFIX,
        ]
    )
else:
    conda_list = (
        "conda/mamba executable not found; "
        "pip freeze retained"
    )


# ------------------------------------------------------------
# GPU metadata
# ------------------------------------------------------------

def nvidia_query(field):
    p = subprocess.run(
        [
            "nvidia-smi",
            f"--query-gpu={field}",
            "--format=csv,noheader,nounits",
        ],
        capture_output=True,
        text=True,
    )

    if p.returncode != 0:
        return None

    value = p.stdout.strip().splitlines()

    return value[0] if value else None


gpu_name = nvidia_query("name")
gpu_vram_total = nvidia_query("memory.total")
gpu_vram_free = nvidia_query("memory.free")


# ------------------------------------------------------------
# Disk metadata
# ------------------------------------------------------------

disk = shutil.disk_usage("/content")

disk_free_gb = round(
    disk.free / 1e9,
    1,
)


# ------------------------------------------------------------
# Merge baseline environment metadata if Cell 1 created it.
# ------------------------------------------------------------

env_pre_current = globals().get(
    "env_pre",
    {},
)

versions = pf.get(
    "versions",
    {},
)

gpu_pf = pf.get(
    "gpu",
    {},
)


env_post = {
    **env_pre_current,

    "env_strategy": (
        "isolated Miniforge Python 3.10 "
        "reproducing upstream install_bindcraft.sh; "
        "Colab system Python/JAX never used for compute"
    ),

    "bindcraft_commit": bindcraft_commit,

    "colabdesign_pinned_commit": globals().get(
        "COLABDESIGN_PIN",
        "unknown",
    ),

    "conda_override_cuda": globals().get(
        "CUDA_OVERRIDE",
        "12.6",
    ),

    "preflight": pf,

    "python_env": versions.get(
        "python"
    ),

    "jax_version": versions.get(
        "jax"
    ),

    "jaxlib_version": versions.get(
        "jaxlib"
    ),

    "numpy_version": versions.get(
        "numpy"
    ),

    "flax_version": versions.get(
        "flax"
    ),

    "colabdesign_version": versions.get(
        "colabdesign"
    ),

    "pyrosetta_installed": bool(
        versions.get("pyrosetta")
    ),

    "jax_devices": gpu_pf.get(
        "all_devices"
    ),

    "xla_backend": pf.get(
        "xla_bridge"
    ),

    "gpu_name_postinstall": gpu_name,

    "gpu_vram_total_mib_postinstall": (
        int(gpu_vram_total)
        if gpu_vram_total
        else None
    ),

    "gpu_vram_free_mib_postinstall": (
        int(gpu_vram_free)
        if gpu_vram_free
        else None
    ),

    "af_weights_source": AF2_URL,

    "af_weights_expected_count": 15,

    "af_weights_files": weight_files,

    "af_weights_validation": (
        "exact expected filename set matched"
    ),

    "disk_free_gb_postinstall": disk_free_gb,
}


# ------------------------------------------------------------
# Persist metadata
# ------------------------------------------------------------

ENV_METADATA_PATH = os.path.join(
    RUNROOT,
    "env_metadata.json"
)

with open(
    ENV_METADATA_PATH,
    "w",
) as f:
    json.dump(
        env_post,
        f,
        indent=2,
    )


PIP_FREEZE_PATH = os.path.join(
    RUNROOT,
    "pip_freeze.txt"
)

with open(
    PIP_FREEZE_PATH,
    "w",
) as f:
    f.write(pip_freeze)


CONDA_LIST_PATH = os.path.join(
    RUNROOT,
    "conda_list.txt"
)

with open(
    CONDA_LIST_PATH,
    "w",
) as f:
    f.write(conda_list)


print(
    "env metadata saved to:",
    ENV_METADATA_PATH,
)

print(
    "\n========================================"
)

print(
    "CELL 7 PASSED"
)

print(
    "AlphaFold weights: 15/15 validated"
)

print(
    "BindCraft commit:",
    bindcraft_commit,
)

print(
    "GPU:",
    gpu_name,
)

print(
    "Free VRAM:",
    gpu_vram_free,
    "MiB",
)

print(
    "========================================"
)

## Cell 8 — write smoke configs into the cloned BindCraft tree

In [ ]:
targets = {
 'pdl1_official_smoke.json': {
    'design_path': f'{RUNROOT}/pdl1_official_smoke/', 'binder_name': 'PDL1_smoke',
    'starting_pdb': '/content/bindcraft/example/PDL1.pdb', 'chains': 'A',
    'target_hotspot_residues': '56', 'lengths': [65, 65], 'number_of_final_designs': 1},
 'egfr_d3_B_conservative.json': {
    'design_path': f'{RUNROOT}/egfr_d3_B_conservative/', 'binder_name': 'EGFR_D3_Bcons',
    'starting_pdb': '/content/egfr/6ARU_chainA_domain3_310-481.pdb', 'chains': 'A',
    'target_hotspot_residues': '390,393,399,421,424,431', 'lengths': [80, 80],
    'number_of_final_designs': 1},
}
os.makedirs('/content/bindcraft/settings_target', exist_ok=True)
for name, d in targets.items():
    json.dump(d, open(f'/content/bindcraft/settings_target/{name}', 'w'), indent=4)

def smoke_advanced(max_traj):
    base = json.load(open('/content/bindcraft/settings_advanced/default_4stage_multimer.json'))
    assert base['max_trajectories'] is False and base['predict_initial_guess'] is False
    base['max_trajectories'] = max_traj
    return base

json.dump(smoke_advanced(1), open('/content/bindcraft/advanced_smoke_max1.json', 'w'), indent=4)
json.dump(smoke_advanced(3), open('/content/bindcraft/advanced_smoke_max3.json', 'w'), indent=4)
default = json.load(open('/content/bindcraft/settings_advanced/default_4stage_multimer.json'))
for f, n in [('advanced_smoke_max1.json', 1), ('advanced_smoke_max3.json', 3)]:
    d2 = json.load(open(f'/content/bindcraft/{f}'))
    diff = {k: (default[k], d2[k]) for k in default if default[k] != d2[k]}
    assert diff == {'max_trajectories': (False, n)}, diff
print('configs written; only max_trajectories differs from official defaults')

## Cell 9 — timed run helper + official PDL1 example (environment test)

In [ ]:
import threading

def _poll_vram(path, stop):
    with open(path, 'w') as f:
        while not stop.is_set():
            used = gpu_query('memory.used')
            f.write(f'{time.time():.0f},{used}\n'); f.flush(); time.sleep(2)

def run_job(tag, settings, advanced):
    log, vram = f'/content/{tag}.log', f'/content/{tag}_vram.csv'
    stop = threading.Event()
    th = threading.Thread(target=_poll_vram, args=(vram, stop)); th.start()
    t0 = time.time()
    with open(log, 'w') as lf:
        p = subprocess.run([BINDPY, '-u', 'bindcraft.py',
                            '--settings', settings,
                            '--filters', './settings_filters/default_filters.json',
                            '--advanced', advanced],
                           cwd='/content/bindcraft', stdout=lf,
                           stderr=subprocess.STDOUT, text=True, env=job_env)
    wall = time.time() - t0; stop.set(); th.join()
    vals = [int(l.split(',')[1]) for l in open(vram) if ',' in l and l.split(',')[1].strip().isdigit()]
    rec = {'tag': tag, 'returncode': p.returncode, 'wall_s': round(wall, 1),
           'peak_vram_mib': max(vals) if vals else None, 'log': log, 'vram_log': vram,
           'python': BINDPY, 'jax': env_post['jax_version']}
    print(json.dumps(rec, indent=2))
    return rec

job_records = {}
job_records['pdl1_official_smoke'] = run_job(
    'pdl1_official_smoke',
    '/content/bindcraft/settings_target/pdl1_official_smoke.json',
    '/content/bindcraft/advanced_smoke_max1.json')
print('---- last 30 log lines ----')
print('\n'.join(open('/content/pdl1_official_smoke.log').read().splitlines()[-30:]))

## Cell 10 — STOP AND CHECK: PDL1 relaxed trajectory gate

EGFR stays blocked until the official example produces a real relaxed PDB. If this asserts, it is an **environment failure** — do not edit EGFR settings, hotspots, lengths, or filters.

In [ ]:
pdl1_dir = f'{RUNROOT}/pdl1_official_smoke'
relaxed = os.path.exists(f'{pdl1_dir}/Trajectory/Relaxed') and \
          [f for f in os.listdir(f'{pdl1_dir}/Trajectory/Relaxed') if f.endswith('.pdb')]
print('PDL1 relaxed trajectories:', relaxed)
assert relaxed, 'official example did not produce a relaxed trajectory -> environment failure, stop here'

## Cell 11 — micro EGFR run: envelope B, conservative 6-residue hotspot, 80 aa, cap 3

In [ ]:
job_records['egfr_d3_B_conservative'] = run_job(
    'egfr_d3_B_conservative',
    '/content/bindcraft/settings_target/egfr_d3_B_conservative.json',
    '/content/bindcraft/advanced_smoke_max3.json')
print('---- last 40 log lines ----')
print('\n'.join(open('/content/egfr_d3_B_conservative.log').read().splitlines()[-40:]))

## Cell 12 — geometry analysis vs approved envelope B + report bundle

In [ ]:
import importlib.util
spec = importlib.util.spec_from_file_location('analyze_bindcraft_run', '/content/analyze_bindcraft_run.py')
analyzer = importlib.util.module_from_spec(spec); sys.modules['analyze_bindcraft_run'] = analyzer
spec.loader.exec_module(analyzer)

analyses = {}
for tag in ['pdl1_official_smoke', 'egfr_d3_B_conservative']:
    rdir = f'{RUNROOT}/{tag}'
    out = f'{RUNROOT}/{tag}_geometry.json'
    sys.argv = ['analyze_bindcraft_run.py', '--run-dir', rdir, '--out', out]
    try:
        analyzer.main()
        analyses[tag] = json.load(open(out))['summary']
    except Exception as e:
        analyses[tag] = {'analysis_error': repr(e)}

report = {'env': env_post, 'jobs': job_records, 'geometry': analyses,
          'notes': 'completed trajectory != successful binder; official filters decide acceptance'}
json.dump(report, open(f'{RUNROOT}/stage2_smoke_report.json', 'w'), indent=2)
print(json.dumps(analyses, indent=2))
print('REPORT:', f'{RUNROOT}/stage2_smoke_report.json')

## Cell 13 — visual inspection of every completed EGFR trajectory

In [ ]:
import glob, py3Dmol
pdbs = sorted(glob.glob(f'{RUNROOT}/egfr_d3_B_conservative/Trajectory/Relaxed/*.pdb')) + \
       sorted(glob.glob(f'{RUNROOT}/egfr_d3_B_conservative/Trajectory/LowConfidence/*.pdb')) + \
       sorted(glob.glob(f'{RUNROOT}/egfr_d3_B_conservative/Trajectory/Clashing/*.pdb'))
print('showing', len(pdbs), 'trajectories')
for pdb in pdbs:
    v = py3Dmol.view(width=500, height=350)
    v.addModel(open(pdb).read(), 'pdb')
    v.setStyle({'chain': 'A'}, {'cartoon': {'color': '#3c5b6f'}})
    v.setStyle({'chain': 'B'}, {'cartoon': {'color': '#B76E79'}})
    for r in list(range(390, 404)) + list(range(421, 432)):
        v.addStyle({'chain': 'A', 'resi': r}, {'stick': {'color': 'yellow'}})
    v.zoomTo({'chain': 'B'}); v.setTitle(os.path.basename(pdb))
    v.show()

## Done — what to send back

Share the Drive folder `BindCraft/stage2_smoke/` (or at minimum `preflight.json`, `env_metadata.json`, `stage2_smoke_report.json`, both `*.log` files and one relaxed PDB). Keep the runtime running if a quick re-check is needed; otherwise Disconnect and delete runtime frees the GPU.

Do not run the broad hotspot config or increase trajectory counts until the smoke report is reviewed. No paid compute without written approval.